# AQI Exploratory Data Analysis
Load, inspect, and visualise the feature store data before training.

In [ ]:
import sys; sys.path.insert(0, '..')
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import hopsworks
import config

sns.set_theme(style='whitegrid')
plt.rcParams['figure.figsize'] = (14, 5)

In [ ]:
# Load from Feature Store
project = hopsworks.login(api_key_value=config.HOPSWORKS_API_KEY, project=config.HOPSWORKS_PROJECT)
fs = project.get_feature_store()
fg = fs.get_feature_group(name=config.FEATURE_GROUP_NAME, version=config.FEATURE_GROUP_VERSION)
df = fg.read()
df['datetime'] = pd.to_datetime(df['timestamp'], unit='s', utc=True)
df = df.sort_values('datetime').reset_index(drop=True)
print(df.shape)
df.head()

In [ ]:
# Basic statistics
df[['aqi','pm25','pm10','o3','no2','temperature','humidity']].describe()

In [ ]:
# AQI over time
plt.figure()
plt.plot(df['datetime'], df['aqi'], linewidth=0.8, color='steelblue')
plt.title('AQI over time — ' + config.CITY_NAME)
plt.xlabel('Date'); plt.ylabel('AQI')
plt.tight_layout(); plt.show()

In [ ]:
# Hourly pattern
hourly = df.groupby('hour')['aqi'].mean()
hourly.plot(kind='bar', color='coral')
plt.title('Average AQI by hour of day')
plt.xlabel('Hour'); plt.ylabel('Mean AQI')
plt.tight_layout(); plt.show()

In [ ]:
# Monthly pattern
monthly = df.groupby('month')['aqi'].mean()
monthly.plot(kind='bar', color='teal')
plt.title('Average AQI by month')
plt.xlabel('Month'); plt.ylabel('Mean AQI')
plt.tight_layout(); plt.show()

In [ ]:
# Correlation heatmap
cols = ['aqi','pm25','pm10','o3','no2','so2','co','temperature','humidity','wind_speed','pressure']
available = [c for c in cols if c in df.columns]
corr = df[available].corr()
plt.figure(figsize=(10,8))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', center=0)
plt.title('Feature correlation matrix')
plt.tight_layout(); plt.show()

In [ ]:
# AQI distribution by level
def get_level(v):
    label, _ = config.get_aqi_level(v)
    return label

df['level'] = df['aqi'].apply(get_level)
level_counts = df['level'].value_counts()
level_counts.plot(kind='bar', color='slateblue')
plt.title('Distribution of AQI levels')
plt.ylabel('Number of hours')
plt.tight_layout(); plt.show()

In [ ]:
# AQI vs Temperature scatter
plt.scatter(df['temperature'], df['aqi'], alpha=0.3, color='darkorange', s=10)
plt.title('AQI vs Temperature')
plt.xlabel('Temperature (°C)'); plt.ylabel('AQI')
plt.tight_layout(); plt.show()